In [ ]:
import cv2
import numpy as np

# Definindo a linha de contagem
linha_contagem = (50, 450)

# Carregando o modelo de detecção de carros
modelo_carros = cv2.CascadeClassifier('haarcascade_car.xml')

# Definindo o histórico de rastreamento
historico_rastreamento = []

# Função para desenhar o retângulo e a contagem
def desenhar_retangulo_contagem(frame, bbox, id):
    x, y, w, h = bbox
    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(frame, str(id), (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# Função para o loop principal
def processar_video(video):
    contagem_carros = 0
    while True:
        ret, frame = video.read()

        # Convertendo para escala de cinza
        frame_cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Detectando carros
        carros_detectados = modelo_carros.detectMultiScale(frame_cinza, 1.1, 2)

        # Atualizando o histórico de rastreamento
        for bbox in carros_detectados:
            existe_rastreamento = False
            for id, bbox_historico in enumerate(historico_rastreamento):
                if np.array_equal(bbox, bbox_historico):
                    historico_rastreamento[id] = bbox
                    existe_rastreamento = True
                    break
            if not existe_rastreamento:
                historico_rastreamento.append(bbox)
                contagem_carros += 1

        # Excluindo rastros antigos
        historico_rastreamento = [bbox for bbox in historico_rastreamento if bbox is not None]

        # Desenhando retângulos e contagem
        for id, bbox in enumerate(historico_rastreamento):
            desenhar_retangulo_contagem(frame, bbox, id)

        # Mostrando o frame
        cv2.imshow('Rastreamento de Carros', frame)

        # Verificando a tecla 'q' para sair
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# Abrindo o vídeo
video = cv2.VideoCapture('video.mp4')

# Processando o vídeo
processar_video(video)

# Liberando a captura de vídeo
video.release()

# Fechando todas as janelas
cv2.destroyAllWindows()

print(f"Total de carros: {contagem_carros}")